# Tentativo 16 · I risultati

**Che cosa abbiamo fatto**
- **Dataset**: testate politiche, testo estratto con trafilatura (modalità precisione), articoli tra 100 e 1.500 parole
  (entrano interi in 2.048 token), doppioni tolti tra tutte le testate, **una testata per gruppo** Owner/Parent Domain.
- **TF-IDF** (il riferimento): parole e coppie di parole + modello lineare.
- **Embedding congelati**: Qwen3-Embedding (0,6B e 8B), senza addestramento, + lo stesso modello lineare.
- **ModernBERT** addestrato: articoli interi a 2.048 token, tutti gli articoli, 5 epoche.

Ogni modello in due impostazioni: **classificazione a 5 classi** e **regressione sul punteggio** NewsGuard (0–100).
Stessi 5 fold per tutti.

In [ ]:
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import funzioni as F

warnings.filterwarnings("ignore")
OUT = Path("Results/confronto"); OUT.mkdir(parents=True, exist_ok=True)
FONTI = {"TF-IDF": "Results/feature", "Embedding Qwen3 0.6B": "Results/embedding/Qwen3-Embedding-0.6B",
         "Embedding Qwen3 8B": "Results/embedding/Qwen3-Embedding-8B", "ModernBERT 2048": "Results/modernbert"}
T = {nome: pd.read_csv(Path(c) / "previsioni_testate.csv", index_col=0) for nome, c in FONTI.items()
     if (Path(c) / "previsioni_testate.csv").exists()}
t = pd.read_csv("Results/dataset/testate.csv", index_col="domain")
print(f"{t.articoli.sum():,} articoli di {len(t)} testate · per livello: " + ", ".join(f"{l} {n}" for l, n in t.livello.value_counts().sort_index().items()))
print("modelli con i risultati:", ", ".join(T))

## 1. Il confronto

Quattro misure: **F1** sui 5 livelli, **tolerant ±1** (giusto anche se sbaglia di un livello), e l'accuratezza nel dire
se una testata è **affidabile** (punteggio ≥ 60), su tutte le testate e solo sulle estreme (l1, l2, l4, l5). In blu la
classificazione a 5 classi, in arancio la regressione; linee grigie: intervalli al 95%.

In [ ]:
valori, intervalli, differenze, vero = F.misure_modelli(T)
valori.to_csv(OUT / "confronto.csv")
F.grafico_confronto(valori, intervalli, vero, OUT / "confronto.png")

## 2. In numeri
In alto i valori; sotto la differenza da TF-IDF con la stessa impostazione (è chiara se l'intervallo non contiene lo zero).

In [ ]:
display(valori.round(3))
differenze if len(differenze) else "per ora c'è solo il TF-IDF"

## 3. ModernBERT è addestrato abbastanza?

La diagnostica su un fold (fold 2): 6 epoche su tutti gli articoli di training. La loss di training scende molto (il
modello impara); la F1 sul test si stabilizza dalla 4ª–5ª epoca: per questo i 5 fold usano 5 epoche.

In [ ]:
f = Path("Results/modernbert/diagnostica/curva_fold2.csv")
if f.exists():
    c = pd.read_csv(f)
    with plt.rc_context(F.STILE_GRAFICI):
        fig, assi = plt.subplots(1, 2, figsize=(11, 3.4))
        assi[0].plot(c.epoca, c.loss, marker="o", color="#2a78d6"); assi[0].axhline(np.log(5), ls=":", color="#52514e")
        assi[0].set_title("loss di training (··· a caso: ln 5 = 1,61)")
        assi[1].plot(c.epoca, c.test, marker="o", color="#eb6834"); assi[1].set_ylim(0, 0.5)
        assi[1].set_title("F1 sul fold di test, dopo ogni epoca")
        for a in assi:
            a.set_xlabel("epoca")
        plt.tight_layout(); plt.savefig(OUT / "diagnostica.png", dpi=180, bbox_inches="tight"); plt.show()
else:
    print("la diagnostica non c'è ancora")